# NB08：全班能力試驗與數據品管（Proficiency Testing & Statistical Quality Control）

**食品分析實驗 週次 8｜Nielsen's Food Analysis, 6th ed. 對應章節：分析數據處理（Evaluating Analytical Data）、品質保證（Quality Assurance）**

## 學習目標 (Learning Objectives)

完成本筆記本後，你應該能夠：

1. 把全班 5 組共同盲樣（水分、灰分、蛋白質、脂肪）彙整成 proximate 表，並用差減法（by difference）算出碳水化合物，理解為什麼它的不確定度是其他四項的**平方和開根號**。
2. 用公式（而非查表）計算 Grubbs 檢定的臨界值，比較「組內 n=3」與「班級層級 n=5」兩種尺度下離群值檢定力的差異。
3. 用 scipy 與手算兩種方式做一元 ANOVA（one-way ANOVA），拆解重複性（$s_r$）與再現性（$s_R$）。
4. 計算 z-score（指定值＝中位數、$\sigma_{pt}$ 固定），並依 $|z|\le2$／$2<|z|<3$／$|z|\ge3$ 分級。
5. 畫 Shewhart 管制圖，並用程式自動偵測 5 條 Westgard 規則（$1_{3s}$、$2_{2s}$、$R_{4s}$、$4_{1s}$、$10_x$）。
6. 建立蛋白質（Kjeldahl）的量測不確定度預算，算出合併標準不確定度 $u_c$ 與擴充不確定度 $U$（$k=2$）。
7. 用「guard band（防護帶）」規則，對照結果 $\pm U$ 與規格，做出放行／不放行／無法判斷的決策。

## 如何使用本筆記本

- 直接「全部執行」（Run all / 執行階段 → 全部執行）即可跑完整份筆記本，使用內建的**模擬教學資料**（與 W8 課堂 deck 完全相同的數字）。
- 若要換成你們班上實際彙整的數據，`load_data()` 函式支援三種資料來源：
  1. **本機/雲端硬碟路徑**：`load_data("data/nb08_sample.csv")` 或你自己上傳的檔名。
  2. **已發布的 Google 試算表 CSV 連結**：檔案 → 共用 → 發布到網路 → 選 CSV，取得類似
     `https://docs.google.com/spreadsheets/d/e/xxx/pub?output=csv` 的網址，傳入 `load_data(那個網址)`。
  3. **在 Colab 上傳檔案**：執行 Setup 區塊中被註解掉的上傳程式碼（使用 `google.colab.files.upload()`）。
  4. 若以上都沒有提供，`load_data()` 會自動退回使用本筆記本內嵌的模擬資料字串，確保筆記本在任何環境都能獨立執行。

> ⚠️ **本筆記本內建的資料是模擬教學資料（simulated teaching data）**，數字取自 W8 課堂 deck（`lab_decks/w08_proficiency/build_w08.py`），不是真實量測值，僅用於練習統計計算與繪圖。做作業時請換成你們班的實際彙整數據。

## 資料格式 (Data Schema) — 與其他週次筆記本共用

本課程所有週次的原始資料一律採用「長格式」（long format），欄位固定如下：

| 欄位 | 說明 |
|---|---|
| `group` | 組別代號。本週為 `G1`–`G5`（5 組共同盲樣），另有兩個「虛擬組別」：`QC`（管制圖的檢核樣品，跟哪一組做的無關）、`TA`（助教另外做的蛋白質重複性示意數據） |
| `student` | 本週與 `group` 相同：水分/灰分/蛋白質/脂肪是「整組共同一個結果」（3 位同學共用），不像 NB01 有逐生的重複測量 |
| `item` | 量測項目代號：`moisture`（水分）、`ash`（灰分）、`protein`（蛋白質，單次測定）、`fat`（脂肪）、`fat_qc_check`（QC 檢核樣品）、`protein_repeatability`（助教蛋白質重複性） |
| `rep` | 重複次數（水分/灰分/脂肪為 1–3；蛋白質固定為 1，因為每組只有單次測定；`fat_qc_check` 為第幾次 run（1–12）；`protein_repeatability` 為助教第幾次重複（1–3）) |
| `value` | 量測值（本週皆為百分比，見 `unit`） |
| `unit` | `value` 的單位（本週固定為 `%`）|
| `temp_C` | 本週不適用（僅 NB01 的體積校正需要水溫），留空 |
| `note` | 備註欄，如：疑似離群值、系統性偏低、懷疑萃取不完全等（可空白）|

## 1. 環境設定 (Setup)

匯入需要的套件，並定義 `load_data()`：一個可同時在 Colab 與本機執行、且不需要額外檔案就能跑的資料載入函式（內建模擬資料以字串形式嵌入本筆記本）。

**注意**：所有圖表的座標軸標籤與標題一律使用**英文**，避免 Colab 預設字型無法顯示中文而出現方框亂碼（tofu）。文字說明則使用繁體中文。

In [ ]:
import io
import os

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

plt.style.use("ggplot")
plt.rcParams["axes.unicode_minus"] = False
pd.set_option("display.precision", 4)

RNG_SEED = 42
np.random.seed(RNG_SEED)

In [ ]:
# ---- Embedded sample data (fallback) ----------------------------------
# This is the exact content of data/nb08_sample.csv, embedded so the
# notebook is self-contained (works even without the data/ folder present).
# Numbers reproduce lab_decks/w08_proficiency/build_w08.py exactly.
SAMPLE_CSV = """group,student,item,rep,value,unit,temp_C,note
G1,G1,moisture,1,2.75,%,,
G1,G1,moisture,2,2.78,%,,
G1,G1,moisture,3,2.8,%,,
G1,G1,ash,1,5.95,%,,
G1,G1,ash,2,6.02,%,,
G1,G1,ash,3,5.98,%,,
G1,G1,protein,1,25.93,%,,
G1,G1,fat,1,26.6,%,,
G1,G1,fat,2,26.4,%,,
G1,G1,fat,3,26.5,%,,
G2,G2,moisture,1,2.85,%,,
G2,G2,moisture,2,2.88,%,,
G2,G2,moisture,3,2.83,%,,
G2,G2,ash,1,6.1,%,,
G2,G2,ash,2,6.05,%,,
G2,G2,ash,3,6.08,%,,
G2,G2,protein,1,26.17,%,,
G2,G2,fat,1,24.6,%,,懷疑 Soxhlet 萃取不完全 (suspected incomplete Soxhlet extraction)
G2,G2,fat,2,24.3,%,,懷疑 Soxhlet 萃取不完全 (suspected incomplete Soxhlet extraction)
G2,G2,fat,3,24.8,%,,懷疑 Soxhlet 萃取不完全 (suspected incomplete Soxhlet extraction)
G3,G3,moisture,1,2.7,%,,
G3,G3,moisture,2,2.74,%,,
G3,G3,moisture,3,3.3,%,,疑似離群值 (possible outlier)
G3,G3,ash,1,5.9,%,,
G3,G3,ash,2,5.95,%,,
G3,G3,ash,3,5.92,%,,
G3,G3,protein,1,26.03,%,,
G3,G3,fat,1,26.3,%,,
G3,G3,fat,2,26.5,%,,
G3,G3,fat,3,26.4,%,,
G4,G4,moisture,1,2.9,%,,
G4,G4,moisture,2,2.92,%,,
G4,G4,moisture,3,2.89,%,,
G4,G4,ash,1,6.05,%,,
G4,G4,ash,2,6.0,%,,
G4,G4,ash,3,6.03,%,,
G4,G4,protein,1,24.47,%,,"系統性偏低 (systematic bias, ~1.5 個百分點)"
G4,G4,fat,1,26.7,%,,
G4,G4,fat,2,26.5,%,,
G4,G4,fat,3,26.6,%,,
G5,G5,moisture,1,2.79,%,,
G5,G5,moisture,2,2.81,%,,
G5,G5,moisture,3,2.8,%,,
G5,G5,ash,1,5.98,%,,
G5,G5,ash,2,6.01,%,,
G5,G5,ash,3,5.99,%,,
G5,G5,protein,1,26.0,%,,
G5,G5,fat,1,26.45,%,,
G5,G5,fat,2,26.6,%,,
G5,G5,fat,3,26.4,%,,
QC,QC,fat_qc_check,1,25.95,%,,
QC,QC,fat_qc_check,2,26.15,%,,
QC,QC,fat_qc_check,3,25.85,%,,
QC,QC,fat_qc_check,4,26.2,%,,
QC,QC,fat_qc_check,5,25.9,%,,
QC,QC,fat_qc_check,6,26.05,%,,
QC,QC,fat_qc_check,7,26.1,%,,
QC,QC,fat_qc_check,8,26.95,%,,
QC,QC,fat_qc_check,9,25.8,%,,
QC,QC,fat_qc_check,10,25.3,%,,
QC,QC,fat_qc_check,11,25.25,%,,
QC,QC,fat_qc_check,12,26.0,%,,
TA,TA,protein_repeatability,1,25.95,%,,
TA,TA,protein_repeatability,2,26.15,%,,
TA,TA,protein_repeatability,3,25.85,%,,
"""

In [ ]:
def load_data(source=None):
    """Load NB08 proficiency-testing / QC data in the shared long-format schema:
    group, student, item, rep, value, unit, temp_C, note

    Parameters
    ----------
    source : str or None
        - None: try a local "data/nb08_sample.csv" relative path first,
          otherwise fall back to the embedded SAMPLE_CSV string.
        - a local file path: read with pandas.read_csv.
        - an http(s) URL (e.g. a published Google Sheets CSV link): read
          directly with pandas.read_csv.

    Returns
    -------
    pandas.DataFrame
    """
    if source is not None:
        try:
            return pd.read_csv(source)
        except Exception as e:
            print(f"讀取 {source} 失敗，改用內建模擬資料。錯誤訊息：{e}")

    local_path = os.path.join("data", "nb08_sample.csv")
    if os.path.exists(local_path):
        return pd.read_csv(local_path)

    return pd.read_csv(io.StringIO(SAMPLE_CSV))


df = load_data()
print(f"讀入 {len(df)} 筆資料，共 {df['group'].nunique()} 個組別代號、"
      f"{df['item'].nunique()} 個量測項目。")
df.head()

### （選用）在 Colab 上傳班級實際資料

若在 Google Colab 中執行、且要換成全班實際彙整值，取消下方程式碼的註解後執行即可；在非 Colab 環境（例如本機 Jupyter）執行這個 cell 不會有任何作用。

In [ ]:
# try:
#     from google.colab import files
#     uploaded = files.upload()
#     fname = list(uploaded.keys())[0]
#     df = load_data(fname)
#     print(f"已改用上傳檔案：{fname}")
# except ImportError:
#     print("非 Colab 環境，略過上傳步驟；請改用 load_data('本機路徑或URL')")

In [ ]:
GROUPS = ["G1", "G2", "G3", "G4", "G5"]


def get_series(data, group, item):
    """Return the `value` column for one (group, item) as a numpy array,
    in `rep` order."""
    sub = data[(data["group"] == group) & (data["item"] == item)].sort_values("rep")
    return sub["value"].to_numpy(dtype=float)

## 2. Proximate 組成表與差減法碳水化合物 (Proximate Table & Carbohydrate by Difference)

把 5 組的水分、灰分、蛋白質、脂肪彙整成一張表。**碳水化合物用差減法（by difference）計算**，不是直接測出來的：

$$\text{碳水化合物\%} = 100 - (\text{水分\%} + \text{灰分\%} + \text{蛋白質\%} + \text{脂肪\%})$$

差減法沒有自己的實驗步驟——它只是拿別人的結果做加減。因此水分、灰分、蛋白質、脂肪**各自的不確定度都會被帶進來**，而且要用「平方和開根號」合併（不確定度不能直接相加）：

$$u_{\text{碳水}} = \sqrt{u_{\text{水分}}^2 + u_{\text{灰分}}^2 + u_{\text{蛋白質}}^2 + u_{\text{脂肪}}^2}$$

蛋白質的標準不確定度 $u_{\text{蛋白質}}$ 需要一份完整的量測不確定度預算（詳見第 7 節），這裡先把它算出來備用（第 7 節會用同一組數字，做完整的分量拆解與說明）。

In [ ]:
moist_mean = {g: round(get_series(df, g, "moisture").mean(), 4) for g in GROUPS}
ash_mean = {g: round(get_series(df, g, "ash").mean(), 4) for g in GROUPS}
fat_mean = {g: round(get_series(df, g, "fat").mean(), 4) for g in GROUPS}
prot_mean = {g: round(get_series(df, g, "protein")[0], 4) for g in GROUPS}  # n=1，單次測定
carb_mean = {
    g: round(100.0 - (moist_mean[g] + ash_mean[g] + prot_mean[g] + fat_mean[g]), 4)
    for g in GROUPS
}

proximate_df = pd.DataFrame({
    "group": GROUPS,
    "moisture_%": [moist_mean[g] for g in GROUPS],
    "ash_%": [ash_mean[g] for g in GROUPS],
    "protein_%": [prot_mean[g] for g in GROUPS],
    "fat_%": [fat_mean[g] for g in GROUPS],
    "carbohydrate_%": [carb_mean[g] for g in GROUPS],
})
proximate_df

In [ ]:
# 蛋白質的「指定值」(assigned/reference value)：5 組單次測定值的中位數。
# 這個數字之後在第 5 節(z-score)會再用到一次(同一個數字，兩種用途)。
protein_reference_value = float(np.median([prot_mean[g] for g in GROUPS]))
print(f"蛋白質指定值 (5 組中位數) = {protein_reference_value:.4f}%")

# ---- 蛋白質的量測不確定度預算 (完整推導與說明在第 7 節；這裡先算出數值供 u_carb 使用) ----
u_weigh_rel = (0.0002 / np.sqrt(3)) / 1.0000 * 100          # 秤重: 分析天平 +-0.0002 g，矩形分布
u_titrant_rel = 0.30                                         # 滴定劑濃度: 標定後的標準不確定度(已給定)
u_vol_rel = (0.05 / np.sqrt(3)) / 25.0 * 100                # 滴定體積: 滴定管刻度 +-0.05 mL，矩形分布
u_factor_rel = (0.03 / np.sqrt(3)) / 6.38 * 100             # 轉換因子: 6.38 +- 0.03，矩形分布

ta_protein_reps = get_series(df, "TA", "protein_repeatability")
s_r_protein = float(np.std(ta_protein_reps, ddof=1))         # 重複性標準差(助教示範3重複)
u_repeat_rel = (s_r_protein / protein_reference_value) * 100  # 單次測定，不除以 sqrt(n)

u_components_rel = {
    "秤重 weighing": u_weigh_rel,
    "滴定劑濃度 titrant conc.": u_titrant_rel,
    "滴定體積 titration volume": u_vol_rel,
    "轉換因子 6.38": u_factor_rel,
    "重複性 repeatability": u_repeat_rel,
}
u_c_rel = float(np.sqrt(sum(v ** 2 for v in u_components_rel.values())))
u_c_abs = protein_reference_value * u_c_rel / 100.0
U_expanded = 2 * u_c_abs  # k=2

print(f"蛋白質 u_c(相對) = {u_c_rel:.4f}%   u_c(絕對) = {u_c_abs:.4f}%   U(k=2) = {U_expanded:.4f}%")

In [ ]:
# 其他三項的標準不確定度 (方法精密度，示意值，取自 W2/W3/W6 的重現性資料，此處直接給定)
u_moist_abs = 0.05
u_ash_abs = 0.08
u_fat_abs = 0.35   # Soxhlet 變異較大

u_carb_abs = float(np.sqrt(u_moist_abs ** 2 + u_ash_abs ** 2 + u_c_abs ** 2 + u_fat_abs ** 2))
U_carb_expanded = 2 * u_carb_abs

print("== 碳水化合物不確定度傳播 u_carb = sqrt(Sigma u^2) ==")
print(f"u_moist={u_moist_abs:.3f}  u_ash={u_ash_abs:.3f}  u_protein={u_c_abs:.3f}  u_fat={u_fat_abs:.3f}")
print(f"  ->  u_carb = {u_carb_abs:.4f}%   U_carb(k=2) = {U_carb_expanded:.4f}%")

可以看到，碳水化合物的不確定度（$u_{\text{碳水}}$）遠大於水分或灰分單獨的不確定度——這正是差減法「繼承所有人誤差」的代價，也是為什麼碳水化合物永遠是 proximate 分析中不確定度最大的一項。

## 3. Grubbs 離群值檢定 (Grubbs' Test for Outliers)

Grubbs 統計量：

$$G = \frac{\max_i |x_i - \bar{x}|}{s}$$

雙側臨界值可以用 t 分布的公式直接算出來（不用查表）：

$$G_{crit} = \frac{n-1}{\sqrt{n}}\sqrt{\frac{t^2_{\alpha/(2n),\,n-2}}{n-2+t^2_{\alpha/(2n),\,n-2}}}$$

### 為什麼 n=3 時幾乎測不出離群值？

$G$ 有一個**理論上界**：不管離群值多誇張，$G$ 都不可能超過

$$G_{max} = \frac{n-1}{\sqrt{n}}$$

當 $n=3$ 時，$G_{max} = \frac{2}{\sqrt{3}} \approx 1.155$，而 $G_{crit}(n=3,\alpha=0.05)$ 剛好就卡在這條線附近——代表**只有 3 個重複值時，這個檢定實質上永遠判不出顯著**，這也是為什麼水分/灰分/脂肪這種「組內三重複」的離群值檢定力很弱。

In [ ]:
def grubbs_crit(n, alpha=0.05):
    """兩側 Grubbs 臨界值:
    G_crit = (n-1)/sqrt(n) * sqrt(t^2/(n-2+t^2))，t = t.ppf(1-alpha/(2n), n-2)"""
    t = stats.t.ppf(1 - alpha / (2 * n), n - 2)
    return (n - 1) / np.sqrt(n) * np.sqrt(t ** 2 / (n - 2 + t ** 2))


Gcrit3 = grubbs_crit(3)
Gcrit5 = grubbs_crit(5)
Gmax3 = 2 / np.sqrt(3)   # (n-1)/sqrt(n) at n=3, 理論上界
Gmax5 = 4 / np.sqrt(5)   # n=5 的理論上界

print(f"G_crit(n=3, a=.05) = {Gcrit3:.4f}   理論上界 (n-1)/sqrt(n) = {Gmax3:.4f}")
print(f"G_crit(n=5, a=.05) = {Gcrit5:.4f}   理論上界 (n-1)/sqrt(n) = {Gmax5:.4f}")

### 3.1 組內層級 (within-group, n=3)：G3 水分的三重複

In [ ]:
g3_moist = get_series(df, "G3", "moisture")
g3_mean, g3_sd = g3_moist.mean(), g3_moist.std(ddof=1)
g3_G = float(np.max(np.abs(g3_moist - g3_mean)) / g3_sd)

print(f"G3 水分三重複: {g3_moist}")
print(f"mean={g3_mean:.4f}  sd={g3_sd:.4f}  G={g3_G:.4f}  vs G_crit(n=3)={Gcrit3:.4f}  (理論上界={Gmax3:.4f})")
if g3_G < Gcrit3:
    print("=> G < G_crit，統計上『測不出來』——即使肉眼看 3.30% 很像離群值。")
else:
    print("=> G >= G_crit，判定為離群值。")

### 3.2 班級層級 (class-level, n=5)：5 組蛋白質單次測定值

In [ ]:
prot_arr = np.array([prot_mean[g] for g in GROUPS])
prot_grand_mean, prot_grand_sd = prot_arr.mean(), prot_arr.std(ddof=1)
prot_G = {g: float(abs(prot_mean[g] - prot_grand_mean) / prot_grand_sd) for g in GROUPS}
flagged_group = max(prot_G, key=prot_G.get)

grubbs_class_df = pd.DataFrame({
    "group": GROUPS,
    "protein_%": [prot_mean[g] for g in GROUPS],
    "G": [round(prot_G[g], 4) for g in GROUPS],
})
print(f"grand mean={prot_grand_mean:.4f}  sd={prot_grand_sd:.4f}")
print(grubbs_class_df)
print(f"\nflagged: {flagged_group}  G={prot_G[flagged_group]:.4f}  vs G_crit(n=5)={Gcrit5:.4f}")
if prot_G[flagged_group] > Gcrit5:
    print(f"=> {flagged_group} 的 G 超過臨界值，判定為離群值。")

## 4. 一元 ANOVA (One-Way ANOVA) — 以灰分為例

用一元變異數分析拆解「組內變異」與「組間變異」，並算出重複性 $s_r$、組間(實驗室間) $s_L$、再現性 $s_R$：

$$s_r = \sqrt{MSW} \qquad s_L = \sqrt{\max\!\big(0,\ \tfrac{MSB-MSW}{n}\big)} \qquad s_R = \sqrt{s_r^2 + s_L^2}$$

In [ ]:
ash_data = [get_series(df, g, "ash") for g in GROUPS]
F_scipy, p_scipy = stats.f_oneway(*ash_data)

k, n_rep = len(GROUPS), 3
N = k * n_rep
grand_mean_ash = float(np.mean([x for arr in ash_data for x in arr]))
SSB = n_rep * sum((arr.mean() - grand_mean_ash) ** 2 for arr in ash_data)
SSW = sum(((arr - arr.mean()) ** 2).sum() for arr in ash_data)
dfB, dfW = k - 1, N - k
MSB, MSW = SSB / dfB, SSW / dfW
F_manual = MSB / MSW
p_manual = float(stats.f.sf(F_manual, dfB, dfW))
s_r = float(np.sqrt(MSW))
s_L = float(np.sqrt(max(0.0, (MSB - MSW) / n_rep)))
s_R = float(np.sqrt(s_r ** 2 + s_L ** 2))
F_crit_ash = float(stats.f.ppf(0.95, dfB, dfW))

print("== ANOVA（灰分, k=5 組, n=3 重複）==")
print(f"手算: SSB={SSB:.5f}  SSW={SSW:.5f}  dfB={dfB}  dfW={dfW}  MSB={MSB:.5f}  MSW={MSW:.5f}")
print(f"F(手算)={F_manual:.4f}  F(scipy)={F_scipy:.4f}  p={p_manual:.4f}  F_crit(0.05)={F_crit_ash:.4f}")
print(f"s_r(重複性)={s_r:.4f}  s_L(組間)={s_L:.4f}  s_R(再現性)={s_R:.4f}")

F 遠大於臨界值、p < 0.05：5 組灰分平均值之間的差異，不能只用組內的重複性（$s_r$）來解釋——代表組間存在真實的系統性落差。這正是為什麼再現性 $s_R$ 一定要考慮組間變異，不能只看自己組的重複測定。

## 5. z-score 與分級 (z-scores & Classification)

$$z = \frac{x_{\text{組}} - x_{\text{assigned}}}{\sigma_{pt}}$$

- $x_{\text{assigned}}$：指定值，用**中位數**（穩健統計量，不容易被離群組拉走）。
- $\sigma_{pt}$：事先訂定的目標標準差（fitness-for-purpose，本課採**固定值** 0.40%，不是用受汙染的班級 SD）。
- 分級規則：$|z|\le2$ 滿意（satisfactory）；$2<|z|<3$ 問題（questionable）；$|z|\ge3$ 不滿意（unsatisfactory）。

In [ ]:
z_assigned = protein_reference_value   # 與第 2 節算出的指定值是同一個數字
sigma_pt = 0.40
z_score = {g: round((prot_mean[g] - z_assigned) / sigma_pt, 3) for g in GROUPS}


def classify_z(z):
    if abs(z) <= 2:
        return "satisfactory"
    if abs(z) < 3:
        return "questionable"
    return "unsatisfactory"


z_df = pd.DataFrame({
    "group": GROUPS,
    "protein_%": [prot_mean[g] for g in GROUPS],
    "z": [z_score[g] for g in GROUPS],
    "category": [classify_z(z_score[g]) for g in GROUPS],
})
print(f"x_assigned = {z_assigned:.4f}%   sigma_pt = {sigma_pt:.2f}%")
z_df

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = ["tab:red" if abs(z_score[g]) >= 3 else ("tab:orange" if abs(z_score[g]) > 2 else "tab:green")
          for g in GROUPS]
ax.bar(GROUPS, [z_score[g] for g in GROUPS], color=colors)
for lim, style in [(2, "--"), (-2, "--"), (3, ":"), (-3, ":")]:
    ax.axhline(lim, color="grey", linestyle=style, linewidth=1)
ax.axhline(0, color="black", linewidth=1)
ax.set_ylabel("z-score (protein)")
ax.set_title("Protein z-scores by group (dashed = +-2, dotted = +-3)")
plt.tight_layout()
plt.show()

## 6. Shewhart 管制圖與 Westgard 規則 (Shewhart Control Chart & Westgard Rules)

QC 檢核樣品的 12 次分析 run。**管制界線用「既有基準」**（歷史建立的 mean / s，不是從這 12 點自己算出來的——否則違規點會被界線自己吃掉）。

本節用程式自動偵測 5 條常見的 Westgard 規則：

- **$1_{3s}$**：任一點超出 $\pm3s$。
- **$2_{2s}$**：連續兩點同側超出 $\pm2s$。
- **$R_{4s}$**：相鄰兩點的全距（range）超過 $4s$（例如一點在 $+2s$ 以上，下一點在 $-2s$ 以下）。
- **$4_{1s}$**：連續四點同側超出 $\pm1s$。
- **$10_x$**：連續十點落在中心線同一側（不論距離多遠）。

In [ ]:
qc_mean, qc_s = 26.0, 0.30   # 既有基準(歷史值)，非由下面 12 點自算
qc_runs = get_series(df, "QC", "fat_qc_check")
qc_z = [(v - qc_mean) / qc_s for v in qc_runs]

print(f"既有基準: mean={qc_mean:.2f}%  s={qc_s:.2f}%")
for i, (v, z) in enumerate(zip(qc_runs, qc_z), start=1):
    print(f"  Run {i:2d}: value={v:.2f}%  z={z:+.3f}")

In [ ]:
def rule_1_3s(z):
    return [i for i, zi in enumerate(z) if abs(zi) > 3]


def rule_2_2s(z):
    viol = set()
    for i in range(len(z) - 1):
        if abs(z[i]) > 2 and abs(z[i + 1]) > 2 and (z[i] > 0) == (z[i + 1] > 0):
            viol.update([i, i + 1])
    return sorted(viol)


def rule_R_4s(z):
    return [i for i in range(len(z) - 1) if abs(z[i] - z[i + 1]) > 4]


def rule_4_1s(z):
    viol = set()
    for i in range(len(z) - 3):
        window = z[i:i + 4]
        if all(abs(w) > 1 for w in window) and (all(w > 0 for w in window) or all(w < 0 for w in window)):
            viol.update(range(i, i + 4))
    return sorted(viol)


def rule_10_x(z):
    viol = set()
    for i in range(len(z) - 9):
        window = z[i:i + 10]
        if all(w > 0 for w in window) or all(w < 0 for w in window):
            viol.update(range(i, i + 10))
    return sorted(viol)


westgard_results = {
    "1_3s": rule_1_3s(qc_z),
    "2_2s": rule_2_2s(qc_z),
    "R_4s": rule_R_4s(qc_z),
    "4_1s": rule_4_1s(qc_z),
    "10_x": rule_10_x(qc_z),
}

print("== Westgard 規則自動偵測結果（違規點的 run 編號，1-based）==")
for rule, idx in westgard_results.items():
    runs = [i + 1 for i in idx]
    status = f"違規於 run {runs}" if runs else "沒有違規"
    print(f"  {rule:5s}: {status}")

在這組 QC 資料中，只有 **$1_{3s}$**（第 8 次 run，突然衝到 26.95%）與 **$2_{2s}$**（第 10、11 次連續兩點都低於 $-2s$）被觸發；$R_{4s}$、$4_{1s}$、$10_x$ 都沒有被觸發。這是刻意設計的教學重點：**不同的 Westgard 規則對「不同型態的異常」敏感度不同**——$1_{3s}$ 抓的是單次劇烈失控，$2_{2s}$ 抓的是短期系統性偏移，而 $4_{1s}$/$10_x$ 是用來抓「緩慢飄移」的規則，這批資料裡沒有那種緩慢趨勢，所以不會觸發。

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
runs_x = np.arange(1, len(qc_runs) + 1)
all_viol = sorted(set(i for idx in westgard_results.values() for i in idx))
point_colors = ["tab:red" if i in all_viol else "tab:blue" for i in range(len(qc_runs))]

ax.plot(runs_x, qc_runs, color="tab:blue", linewidth=1.5, zorder=2)
ax.scatter(runs_x, qc_runs, c=point_colors, s=[70 if i in all_viol else 35 for i in range(len(qc_runs))], zorder=3)

for k_, style, color in [(0, "-", "black"), (2, "--", "tab:orange"), (3, ":", "tab:red")]:
    if k_ == 0:
        ax.axhline(qc_mean, color=color, linestyle=style, linewidth=1.2, label="mean")
    else:
        ax.axhline(qc_mean + k_ * qc_s, color=color, linestyle=style, linewidth=1.2, label=f"+/-{k_}s")
        ax.axhline(qc_mean - k_ * qc_s, color=color, linestyle=style, linewidth=1.2)

ax.set_xlabel("Run number")
ax.set_ylabel("QC value (%)")
ax.set_title("Shewhart control chart: QC check sample (established mean/s)")
ax.legend(loc="best", fontsize=8)
plt.tight_layout()
plt.show()

## 7. 蛋白質量測不確定度預算 (Measurement Uncertainty Budget — Protein)

這裡把第 2 節已經算過的數字重新攤開，逐一說明每個分量的來源。Kjeldahl 蛋白質測定的不確定度來源：

| 來源 | 說明 |
|---|---|
| 秤重 (weighing) | 分析天平 $\pm0.0002$ g，矩形分布，標準不確定度 $= \dfrac{\text{半寬}}{\sqrt3}$ |
| 滴定劑濃度 (titrant conc.) | 0.1 N HCl 標定後的標準不確定度（已經是標準差型式，直接使用） |
| 滴定體積 (titration volume) | 50 mL 滴定管刻度 $\pm0.05$ mL，矩形分布 |
| 轉換因子 (conversion factor) | 牛乳蛋白質係數 $6.38\pm0.03$，矩形分布 |
| 重複性 (repeatability) | 每組蛋白質**只有單次測定**（n=1，見 W5 消化爐 8 管限制），無法用組內重複反推 $s_r$；改用助教另外對同一份奶粉做的 3 重複測定，得到方法的 $s_r$。**因為每組報告的就是這一個單次值（未再平均），重複性直接以 $s_r$ 貢獻，不再除以 $\sqrt{n}$**。 |

矩形分布（儀器刻度/公差）的標準不確定度 = 半寬 ÷ $\sqrt3$；已經是標準差型式的分量（如滴定劑標定）則直接使用。合併與擴充：

$$u_c = \sqrt{\sum (\text{各分量})^2} \qquad U = k \cdot u_c \ \ (k=2,\ \approx95\%\text{信心水準})$$

In [ ]:
budget_df = pd.DataFrame({
    "component": list(u_components_rel.keys()),
    "relative_std_uncertainty_%": [round(v, 4) for v in u_components_rel.values()],
})
print(f"助教 3 重複測定值: {ta_protein_reps}  ->  s_r(重複性) = {s_r_protein:.4f}%")
budget_df

In [ ]:
print(f"合併相對標準不確定度 u_c(rel) = {u_c_rel:.4f}%")
print(f"指定值 (reference value)      = {protein_reference_value:.4f}%")
print(f"合併絕對標準不確定度 u_c(abs) = {u_c_abs:.4f}%")
print(f"擴充不確定度 U (k=2)          = {U_expanded:.4f}%")
print()
print(f"=> 報告寫法：蛋白質 = {protein_reference_value:.2f}% +/- {U_expanded:.2f}%  (U, k=2)")

## 8. 放行判定：Guard Band 規則 (Release Decision with Guard Band)

回到開場的問題：奶油測得 **80.3% ± 0.4%**（U, k=2），規格是乳脂 $\ge80\%$，這批**可以放行**嗎？

**guard band（防護帶）規則**（本課採用）：

- 規格是「$\ge$ 下限」：只有當**最壞情況（結果 $-U$）$\ge$ 規格**才判定**放行**；當**最佳情況（結果 $+U$）$<$ 規格**時判定**不放行**；其餘（規格落在結果$\pm U$ 區間內）判定**無法判斷**。
- 規格是「$\le$ 上限」：只有當**最壞情況（結果 $+U$）$\le$ 規格**才判定**放行**；當**最佳情況（結果 $-U$）$>$ 規格**時判定**不放行**；其餘判定**無法判斷**。

「測到的數字比規格高（或低）」跟「這批貨真的合格」不是同一件事——不確定度存在的意義，就是讓「合格」不再只靠一個點估計數字說了算。

In [ ]:
def release_decision(result, U, spec, spec_type):
    """spec_type: ">=" (至少要達到規格，如乳脂/蛋白質下限)
                  "<=" (不能超過規格，如水分/鈉上限)"""
    if spec_type == ">=":
        worst, best = result - U, result + U
        if worst >= spec:
            return "放行 (pass)", worst
        if best < spec:
            return "不放行 (fail)", worst
        return "無法判斷 (inconclusive)", worst
    elif spec_type == "<=":
        worst, best = result + U, result - U
        if worst <= spec:
            return "放行 (pass)", worst
        if best > spec:
            return "不放行 (fail)", worst
        return "無法判斷 (inconclusive)", worst
    raise ValueError("spec_type must be '>=' or '<='")


release_cases = pd.DataFrame([
    {"case": "奶油 乳脂 (butter fat)", "result": 80.3, "U": 0.4, "spec": 80.0, "spec_type": ">="},
    {"case": "全脂奶粉 蛋白質 (milk powder protein)", "result": 26.5, "U": 0.3, "spec": 25.0, "spec_type": ">="},
    {"case": "全脂奶粉 水分 (milk powder moisture)", "result": 3.6, "U": 0.15, "spec": 3.5, "spec_type": "<="},
    {"case": "罐頭湯 鈉 (canned soup sodium, mg/100g)", "result": 620, "U": 15, "spec": 500, "spec_type": "<="},
    {"case": "果汁 維生素C (juice vitamin C, mg/100mL)", "result": 33.0, "U": 1.0, "spec": 30.0, "spec_type": ">="},
    {"case": "巧克力 水分 (chocolate moisture)", "result": 2.30, "U": 0.10, "spec": 2.0, "spec_type": "<="},
])

decisions, worst_cases = [], []
for _, r in release_cases.iterrows():
    decision, worst = release_decision(r["result"], r["U"], r["spec"], r["spec_type"])
    decisions.append(decision)
    worst_cases.append(round(worst, 4))
release_cases["worst_case"] = worst_cases
release_cases["decision"] = decisions
release_cases

特別留意第一列（奶油）：點估計 80.3% 已經高於規格 80%，但最壞情況 $80.3-0.4=79.9\%$ 跌破規格，所以判定是**無法判斷**，不是「放行」——即使測到的數字看起來合格。

## 9. 練習題 (Exercises)

請在每題下方的空白程式碼區塊中作答（可以用文字 print 說明，也可以搭配計算）。

### 問題 1

如果 G3 水分的第 3 重複值不是 3.30%，而是誤差更大的 3.60%，重新計算 Grubbs 的 $G$ 值。這樣是否仍然「測不出來」（$G < G_{crit}$）？

In [ ]:
# TODO: 請在這裡作答

### 問題 2

在「班級層級 Grubbs」（5 組蛋白質單次測定值）中，除了被標記離群的那一組之外，找出 $G$ 值第二大的組別，並查它的 z-score 落在哪個等級（satisfactory / questionable / unsatisfactory）。

In [ ]:
# TODO: 請在這裡作答

### 問題 3

比較灰分 ANOVA 算出來的 $s_r$（重複性）與 $s_L$（組間）何者較大？這代表什麼意義？如果某次改用更嚴謹的操作，使得 $s_L$ 幾乎降到 0，$s_R$ 會變得跟什麼很接近？

In [ ]:
# TODO: 請在這裡作答

### 問題 4

蛋白質不確定度預算中，哪一個分量貢獻最大？如果能把「滴定劑濃度」的相對標準不確定度從 0.30% 降到 0.10%（重新標定、用更精確的方法），合併不確定度 $u_c$（相對）會變成多少？影響大不大？

In [ ]:
# TODO: 請在這裡作答

### 問題 5

如果巧克力水分的規格從 $\le2\%$ 放寬成 $\le2.5\%$，測得結果仍是 $2.30\% \pm 0.10\%$，用 guard band 規則重新判定，結果會不會跟原規格（$\le2\%$，判定「不放行」）不一樣？

In [ ]:
# TODO: 請在這裡作答

## 10. 匯出結果 (Export)

把彙整後的 proximate 表、Grubbs/ANOVA/z-score/放行判定結果存成 CSV。若在 Colab 環境執行，會另外觸發瀏覽器下載；在本機執行則只會存檔，不會嘗試下載（用 try/except 保護，確保兩種環境都能跑）。

In [ ]:
output_path = "nb08_proximate_results.csv"
proximate_df.merge(z_df[["group", "z", "category"]], on="group").to_csv(output_path, index=False)
print(f"已儲存：{output_path}")

release_output_path = "nb08_release_decisions.csv"
release_cases.to_csv(release_output_path, index=False)
print(f"已儲存：{release_output_path}")

try:
    from google.colab import files
    files.download(output_path)
    files.download(release_output_path)
except ImportError:
    print("非 Colab 環境，略過自動下載（檔案已存在本機/雲端硬碟工作目錄）。")